# TPC Event 3D Viewer

Interactive visualization of the point-cloud CSV produced by `tpcanalysis-main/run_mini.cpp`.

- **event_id**: click the box, then ↑/↓ to step ±1 or type a number directly.
- **voxel / min_count**: voxel-coincidence filter. `voxel=0` disables it (default). Try `voxel=1 mm, min_count=10` for a gentle clean-up, `min_count=20` for aggressive ghost rejection.
- **max pts**: safety cap on rendered markers.
- **n lines**: 0 disables line fitting; >0 runs iterative BFGS and overlays up to N lines. Fitting takes a few seconds per event.
- **fit thresh**: per-point distance (mm) for a point to count as an inlier of a fitted line.

In [1]:
import sys
sys.path.insert(0, '.')
from viewer3d import load_points, plot_event
import ipywidgets as widgets

CSV_PATH = '../0423_0003.csv'
df = load_points(CSV_PATH)
events = sorted(df['event_id'].unique().tolist())
print(f'Loaded {len(df)} points across {len(events)} events (range: {events[0]}..{events[-1]})')

Loaded 108500962 points across 1148 events (range: 0..1147)


In [2]:
event = widgets.BoundedIntText(
    value=events[0], min=events[0], max=events[-1], step=1,
    description='event_id',
)
voxel = widgets.FloatSlider(
    value=0.0, min=0.0, max=5.0, step=0.5, description='voxel [mm]',
)
min_count = widgets.IntSlider(
    value=1, min=1, max=100, step=1, description='min_count',
)
max_points = widgets.IntSlider(
    value=20000, min=1000, max=200000, step=1000, description='max pts',
)
n_lines = widgets.IntSlider(
    value=0, min=0, max=6, step=1, description='n lines',
)
fit_thresh = widgets.FloatSlider(
    value=3.0, min=0.5, max=10.0, step=0.5, description='fit thresh',
)
out = widgets.Output()

def render(change=None):
    out.clear_output(wait=True)
    with out:
        try:
            fig = plot_event(
                df, event.value,
                voxel_size_mm=voxel.value,
                min_count=min_count.value,
                max_points=max_points.value,
                n_lines=n_lines.value,
                fit_threshold_mm=fit_thresh.value,
            )
            fig.show()
        except ValueError as e:
            print(e)

for w in (event, voxel, min_count, max_points, n_lines, fit_thresh):
    w.observe(render, names='value')

widgets.VBox([
    widgets.HBox([event, voxel]),
    widgets.HBox([min_count, max_points]),
    widgets.HBox([n_lines, fit_thresh]),
    out,
])

In [3]:
render()

## Batch fit: all events, lines only

Runs the same fitter on every event in the file and draws **only the fitted lines** on a single 3D figure. Voxel filter and fit parameters come from the sliders above — tweak them first, then hit **Fit all events**. For a 142-event file this takes tens of seconds; for 1000+ events expect minutes.

- **color by**: `event_id` (Viridis over time), `inliers` (Viridis over fit quality), or `none` (single colour, differentiate by opacity).
- **opacity**: lower it when you have 1000+ lines so the interior of the detector stays readable.
- **length [mm]**: endpoint-to-endpoint length window. Drag the bottom up to **80 mm** to isolate cosmic-like full-cage tracks; drag the top down to **30 mm** to isolate short alpha-like tracks. Updates live without refitting.

In [ ]:
from viewer3d import plot_all_lines, plot_length_histogram, filter_dense_voxels
from line_fit import fit_all_events

color_by = widgets.Dropdown(
    options=['event_id', 'inliers', 'none'],
    value='event_id',
    description='color by',
)
opacity = widgets.FloatSlider(
    value=0.4, min=0.05, max=1.0, step=0.05, description='opacity',
)
length_window = widgets.FloatRangeSlider(
    value=[0.0, 250.0], min=0.0, max=250.0, step=5.0,
    description='length [mm]', readout_format='.0f',
)
fit_btn = widgets.Button(description='Fit all events', button_style='primary')
progress = widgets.IntProgress(value=0, min=0, max=len(events), description='progress')
status = widgets.Label(value='idle')
hist_out = widgets.Output()
batch_out = widgets.Output()

# Cache so redrawing with different color/opacity/length doesn't refit.
state = {'pairs': None, 'params': None}

def current_params():
    return (voxel.value, min_count.value, n_lines.value, fit_thresh.value)

def run_fit(_=None):
    params = current_params()
    fit_btn.disabled = True
    status.value = 'fitting…'
    progress.value = 0

    def _prog(done, total):
        progress.max = total
        progress.value = done
        if done % 10 == 0 or done == total:
            status.value = f'fitting… {done}/{total}'

    def _filter(ev):
        return filter_dense_voxels(ev, voxel_size_mm=voxel.value, min_count=min_count.value)

    pairs = fit_all_events(
        df,
        filter_fn=_filter,
        n_lines=max(1, n_lines.value),  # n_lines=0 in slider → fit 1 line per event
        distance_threshold_mm=fit_thresh.value,
        progress=_prog,
    )
    state['pairs'] = pairs
    state['params'] = params
    status.value = f'done: {len(pairs)} lines from {len(events)} events'
    fit_btn.disabled = False
    draw()

def draw(_=None):
    if state['pairs'] is None:
        return
    lo, hi = length_window.value
    hist_out.clear_output(wait=True)
    with hist_out:
        plot_length_histogram(
            state['pairs'], min_length_mm=lo, max_length_mm=hi,
        ).show()
    batch_out.clear_output(wait=True)
    with batch_out:
        plot_all_lines(
            state['pairs'],
            color_by=color_by.value,
            opacity=opacity.value,
            min_length_mm=lo,
            max_length_mm=hi,
        ).show()

fit_btn.on_click(run_fit)
color_by.observe(draw, names='value')
opacity.observe(draw, names='value')
length_window.observe(draw, names='value')

widgets.VBox([
    widgets.HBox([fit_btn, progress, status]),
    widgets.HBox([color_by, opacity]),
    widgets.HBox([length_window]),
    hist_out,
    batch_out,
])